In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from time import time

def size_mb(docs):
    return sum(len(s.encode("utf-8")) for s in docs) / 1e6

# Sample documents
documents = [
    "This is the first document.",
    "This is the second document.",
    "The third document is here.",
    "Is this the first document?",
]

# Initialize TfidfVectorizer with unigrams and bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2))  # (1, 2) for unigrams and bigrams

# "Fit" and "transform" the data
t0 = time()
tfidf_matrix = vectorizer.fit_transform(documents)
duration_train = time() - t0

documents_size_mb = size_mb(documents)
print(
    f"vectorize training done in {duration_train:.3f}s "
    f"at {documents_size_mb / duration_train:.3f}MB/s"
)

# Get the feature names (tokens)
feature_names = vectorizer.get_feature_names_out()

# Convert the TF-IDF matrix to a dense array for easier printing
tfidf_array = tfidf_matrix.toarray()

vectorize training done in 0.003s at 0.037MB/s


In [6]:
feature_names

array(['document', 'document is', 'first', 'first document', 'here', 'is',
       'is here', 'is the', 'is this', 'second', 'second document', 'the',
       'the first', 'the second', 'the third', 'third', 'third document',
       'this', 'this is', 'this the'], dtype=object)

In [7]:
# Create a dictionary mapping documents to their TF-IDF vectors, with tokens as keys
document_tfidf_dict = {
    documents[i]: {feature_names[j]: tfidf_array[i][j] for j in range(len(feature_names))}
    for i in range(len(documents))
}

In [8]:
document_tfidf_dict

{'This is the first document.': {'document': 0.25071358682047334,
  'document is': 0.0,
  'first': 0.3787845111946505,
  'first document': 0.3787845111946505,
  'here': 0.0,
  'is': 0.25071358682047334,
  'is here': 0.0,
  'is the': 0.3787845111946505,
  'is this': 0.0,
  'second': 0.0,
  'second document': 0.0,
  'the': 0.25071358682047334,
  'the first': 0.3787845111946505,
  'the second': 0.0,
  'the third': 0.0,
  'third': 0.0,
  'third document': 0.0,
  'this': 0.3066587069463172,
  'this is': 0.3787845111946505,
  'this the': 0.0},
 'This is the second document.': {'document': 0.22317326520013453,
  'document is': 0.0,
  'first': 0.0,
  'first document': 0.0,
  'here': 0.0,
  'is': 0.22317326520013453,
  'is here': 0.0,
  'is the': 0.3371758876038861,
  'is this': 0.0,
  'second': 0.4276648597051107,
  'second document': 0.4276648597051107,
  'the': 0.22317326520013453,
  'the first': 0.0,
  'the second': 0.4276648597051107,
  'the third': 0.0,
  'third': 0.0,
  'third document':

In [ ]:
# ---  Simple Querying and Pointwise (Not Re-, but Sort of Level 1) Ranking ---
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np # import numpy
def get_relevant_documents(query, document_tfidf_dict, top_n=2):
    """
    Finds the top_n most relevant documents to a given query based on TF-IDF similarity.

    Args:
        query (str): The query string.
        document_tfidf_dict (dict):  Dictionary mapping documents to their TF-IDF vectors.
        top_n (int): The number of top documents to return.

    Returns:
        list: A list of tuples, where each tuple contains (document, similarity_score).
              Sorted in descending order of similarity.  Returns an empty list if no docs
              or query is provided.
    """
    if not document_tfidf_dict or not query:
        return []

    t0 = time()
    query_vector = vectorizer.transform([query]).toarray()[0]  # Get TF-IDF vector for query
    duration_query_transform = time() - t0
    query_size_mb = size_mb(query)
    print(
        f"query vectorized in {duration_query_transform:.3f}s "
        f"at {documents_size_mb / duration_query_transform:.3f}MB/s"
    )

    similarities = []

    for doc, doc_vector_dict in document_tfidf_dict.items():
        doc_vector = [doc_vector_dict[token] for token in feature_names] # convert dict to ordered list
        # Reshape doc_vector to a 2D array with one row using numpy.array
        doc_vector = np.array([doc_vector])
        similarity = cosine_similarity(query_vector.reshape(1, -1), doc_vector)[0][0] # Reshape both to 2D, query_vector too
        similarities.append((doc, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)  # Sort by similarity score
    return similarities[:top_n]  # Return the top_n documents

In [12]:
# Example usage:
query = "his is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first document"
relevant_docs = get_relevant_documents(query, document_tfidf_dict, top_n=3)

print(f"Top 2 documents for query: '{query}'")
for doc, similarity in relevant_docs:
    print(f"- {doc} (Similarity: {similarity:.4f})")

query vectorized in 0.001s at 0.109MB/s
Top 2 documents for query: 'his is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first documenthis is the first document'
- This is the first document. (Similarity: 0.7615)
- Is this the first document? (Similarity: 0.5252)
- This is the second document. (Similarity: 0.3255)
